In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
import os, shutil


DRIVE_INPUT = '/content/drive/MyDrive/AD-FlowTSE_checkpoints/reuse_experiment_input'

LOCAL_INPUT = '/content/reuse_input'
LOCAL_OUTPUT = '/content/reuse_output'

if os.path.exists(DRIVE_INPUT):
    print(f"Found input on Drive: {DRIVE_INPUT}")
    if os.path.exists(LOCAL_INPUT):
        shutil.rmtree(LOCAL_INPUT)
    shutil.copytree(DRIVE_INPUT, LOCAL_INPUT)
    print("Copied to /content/reuse_input/")
else:
    print(f"Not found at {DRIVE_INPUT}")
    print("Please upload reuse_experiment/input/ to your Drive first,")
    print("or adjust DRIVE_INPUT above.")

os.makedirs(LOCAL_OUTPUT, exist_ok=True)


for root, dirs, files in os.walk(LOCAL_INPUT):
    level = root.replace(LOCAL_INPUT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in files[:5]:
        print(f'{subindent}{f}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files)-5} more')


In [ ]:
!pip install -q yt-dlp

import subprocess, os

yt_dir = os.path.join(LOCAL_INPUT, 'youtube')
os.makedirs(yt_dir, exist_ok=True)

YT_URL = 'https://www.youtube.com/watch?v=us8B_KdcB3c'
YT_START = 710  # start at 11:50

raw_path = os.path.join(yt_dir, 'documentary_full.wav')

!yt-dlp -x --audio-format wav --audio-quality 0 \
    -o "{yt_dir}/documentary_full.%(ext)s" \
    "{YT_URL}"

if os.path.exists(raw_path):
    import torchaudio
    wav, sr = torchaudio.load(raw_path)
    dur = wav.shape[-1] / sr
    print(f"Downloaded: {dur:.1f}s at {sr}Hz, channels={wav.shape[0]}")
else:
    for f in os.listdir(yt_dir):
        print(f"  Found: {f}")
    print("Check the file above and rename if needed.")


In [ ]:
import torchaudio
import torch
import os

yt_dir = os.path.join(LOCAL_INPUT, 'youtube')
TARGET_SR = 16000
YT_START = 710


raw_file = None
for f in os.listdir(yt_dir):
    if f.startswith('documentary_full'):
        raw_file = os.path.join(yt_dir, f)
        break

if raw_file is None:
    raise FileNotFoundError("YouTube audio not found. Re-run Cell 4.")

wav, sr = torchaudio.load(raw_file)
print(f"Raw: {wav.shape}, sr={sr}")


if wav.shape[0] > 1:
    wav = wav.mean(dim=0, keepdim=True)


start_sample = int(YT_START * sr)
wav = wav[:, start_sample:]


MAX_DUR = 300
max_samples = int(MAX_DUR * sr)
if wav.shape[-1] > max_samples:
    wav = wav[:, :max_samples]


if sr != TARGET_SR:
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

out_path = os.path.join(yt_dir, 'documentary.wav')
torchaudio.save(out_path, wav, TARGET_SR)
dur = wav.shape[-1] / TARGET_SR
print(f"Saved: {out_path} ({dur:.1f}s, {TARGET_SR}Hz, mono)")


In [ ]:
!pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
print("\n\n=== DONE installing CUDA PyTorch ===")
print("=== Now: Runtime -> Restart session, then SKIP this cell and run Cell 6b below ===")


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q huggingface_hub safetensors librosa soundfile

!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post1/causal_conv1d-1.6.1.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

import mamba_ssm
print(f"\nmamba-ssm version: {mamba_ssm.__version__}")
print("SUCCESS!")


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)

import os
print("\nRE-USE files:")
for f in sorted(os.listdir('./REUSE')):
    print(f"  {f}")


In [ ]:
import os
import sys
import math
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm

sys.path.insert(0, './REUSE')
from models.stfts import mag_phase_stft, mag_phase_istft
from models.generator_SEMamba_time_d4 import SEMamba
from utils.util import load_config, pad_or_trim_to_match
from huggingface_hub import hf_hub_download

RELU = nn.ReLU()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

config_path = hf_hub_download(repo_id='nvidia/RE-USE', filename='config.json')
cfg = load_config(config_path)
n_fft = cfg['stft_cfg']['n_fft']
hop_size = cfg['stft_cfg']['hop_size']
win_size = cfg['stft_cfg']['win_size']
compress_factor = cfg['model_cfg']['compress_factor']
sampling_rate = cfg['stft_cfg']['sampling_rate']

print("Loading RE-USE model...")
SE_model = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_model.eval()
print("Model loaded.")

def make_even(value):
    value = int(round(value))
    return value if value % 2 == 0 else value + 1

def enhance_short(wav_tensor, sr, model, device):
    wav = wav_tensor.to(device)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)

    with torch.no_grad():
        noisy_mag, noisy_pha, _ = mag_phase_stft(
            wav, n_fft=n_fft_s, hop_size=hop_s, win_size=win_s,
            compress_factor=compress_factor, center=True, addeps=False
        )
        amp_g, pha_g, _ = model(noisy_mag, noisy_pha)
        mag = torch.expm1(RELU(amp_g))
        zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
        amp_g[:, :, (zero_portion > 0.5)[0]] = 0
        audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
        audio_g = pad_or_trim_to_match(wav.detach(), audio_g, pad_value=1e-8)
    return audio_g.cpu()

def enhance_chunked(wav_tensor, sr, model, device, chunk_sec=5.0, hop_portion=0.5):
    wav = wav_tensor.to(device)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)

    chunk_size = int(chunk_sec * sr)
    hop_length = int(hop_portion * chunk_size)
    window = torch.hann_window(chunk_size).to(device)

    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)

    audio_enhanced = torch.zeros_like(wav)
    window_sum = torch.zeros_like(wav)

    n_chunks = max(1, math.ceil((wav.shape[1] - chunk_size) / hop_length) + 1)

    with torch.no_grad():
        for c in range(wav.shape[0]):
            noisy_wav = wav[c:c+1, :]
            for i in tqdm(range(n_chunks), desc=f"  ch{c}", leave=False):
                start = i * hop_length
                chunk = noisy_wav[:, start:start+chunk_size]

                noisy_mag, noisy_pha, _ = mag_phase_stft(
                    chunk, n_fft=n_fft_s, hop_size=hop_s, win_size=win_s,
                    compress_factor=compress_factor, center=True, addeps=False
                )
                amp_g, pha_g, _ = model(noisy_mag, noisy_pha)
                mag = torch.expm1(RELU(amp_g))
                zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
                amp_g[:, :, (zero_portion > 0.5)[0]] = 0

                audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
                audio_g = pad_or_trim_to_match(chunk.detach(), audio_g, pad_value=1e-8)

                seg_len = audio_g.shape[1]
                win_seg = window[:seg_len]
                audio_enhanced[c:c+1, start:start+seg_len] += audio_g * win_seg
                window_sum[c:c+1, start:start+seg_len] += win_seg

    nonzero = window_sum > 1e-8
    audio_enhanced[:, nonzero[0]] = audio_enhanced[:, nonzero[0]] / window_sum[:, nonzero[0]]
    return audio_enhanced.cpu()

LONG_THRESHOLD = 30

def enhance_auto(wav_tensor, sr, model, device):
    dur = wav_tensor.shape[-1] / sr
    if dur > LONG_THRESHOLD:
        return enhance_chunked(wav_tensor, sr, model, device)
    else:
        return enhance_short(wav_tensor, sr, model, device)

print("Enhancement functions ready.")


In [ ]:
import os, torchaudio

yt_in = os.path.join(LOCAL_INPUT, 'youtube', 'documentary.wav')
yt_out_dir = os.path.join(LOCAL_OUTPUT, 'youtube')
os.makedirs(yt_out_dir, exist_ok=True)

wav, sr = torchaudio.load(yt_in)
if wav.shape[0] > 1:
    wav = wav.mean(dim=0, keepdim=True)
print(f"YouTube input: {wav.shape[-1]/sr:.1f}s at {sr}Hz")

enhanced = enhance_auto(wav, sr, SE_model, device)


torchaudio.save(os.path.join(yt_out_dir, 'before.wav'), wav, sr)
torchaudio.save(os.path.join(yt_out_dir, 'after_reuse.wav'), enhanced, sr)
print(f"Saved: {yt_out_dir}/before.wav and after_reuse.wav")


In [ ]:
import os, torchaudio

dihard_in = os.path.join(LOCAL_INPUT, 'dihard')
dihard_out = os.path.join(LOCAL_OUTPUT, 'dihard')
os.makedirs(dihard_out, exist_ok=True)

for fname in sorted(os.listdir(dihard_in)):
    if not fname.endswith('.wav'):
        continue
    rec_id = fname.replace('.wav', '')
    print(f"\nProcessing {rec_id}...")

    wav, sr = torchaudio.load(os.path.join(dihard_in, fname))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    print(f"  Input: {wav.shape[-1]/sr:.1f}s")

    rec_out = os.path.join(dihard_out, rec_id)
    os.makedirs(rec_out, exist_ok=True)

    enhanced = enhance_auto(wav, sr, SE_model, device)

    torchaudio.save(os.path.join(rec_out, 'before.wav'), wav, sr)
    torchaudio.save(os.path.join(rec_out, 'after_reuse.wav'), enhanced, sr)
    print(f"  Saved: {rec_out}/before.wav and after_reuse.wav")

print("\nDIHARD done.")


In [ ]:
import os, shutil, torchaudio
from tqdm import tqdm

lm_in = os.path.join(LOCAL_INPUT, 'librimix')
lm_out = os.path.join(LOCAL_OUTPUT, 'librimix')

for sub in ['before', 'after_reuse', 'clean_gt', 'enrollment']:
    os.makedirs(os.path.join(lm_out, sub), exist_ok=True)

mix_files = sorted(os.listdir(os.path.join(lm_in, 'mixture')))
print(f"Processing {len(mix_files)} LibriMix samples...")

for fname in tqdm(mix_files):
    if not fname.endswith('.wav'):
        continue
    mix_id = fname.replace('.wav', '')

    wav, sr = torchaudio.load(os.path.join(lm_in, 'mixture', fname))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    enhanced = enhance_auto(wav, sr, SE_model, device)

    torchaudio.save(os.path.join(lm_out, 'before', fname), wav, sr)
    torchaudio.save(os.path.join(lm_out, 'after_reuse', fname), enhanced, sr)

    gt_src = os.path.join(lm_in, 'clean_s1', fname)
    enr_src = os.path.join(lm_in, 'enrollment', fname)
    if os.path.exists(gt_src):
        shutil.copy2(gt_src, os.path.join(lm_out, 'clean_gt', fname))
    if os.path.exists(enr_src):
        shutil.copy2(enr_src, os.path.join(lm_out, 'enrollment', fname))

print(f"LibriMix done. Results in {lm_out}/")


In [ ]:
import os
import csv
import torch
import torchaudio
import numpy as np

def si_sdr(estimate, reference):
    estimate = estimate - estimate.mean()
    reference = reference - reference.mean()
    dot = torch.sum(estimate * reference)
    s_target = dot * reference / (torch.sum(reference ** 2) + 1e-8)
    e_noise = estimate - s_target
    return (10 * torch.log10(
        torch.sum(s_target ** 2) / (torch.sum(e_noise ** 2) + 1e-8) + 1e-8
    )).item()

try:
    from pesq import pesq as pesq_fn
    HAS_PESQ = True
    print("PESQ available")
except ImportError:
    HAS_PESQ = False
    print("PESQ not available, installing...")
    !pip install -q pesq
    try:
        from pesq import pesq as pesq_fn
        HAS_PESQ = True
        print("PESQ installed successfully")
    except:
        HAS_PESQ = False
        print("PESQ unavailable, skipping")

try:
    from pystoi import stoi as stoi_fn
    HAS_STOI = True
    print("STOI available")
except ImportError:
    HAS_STOI = False
    print("STOI not available, installing...")
    !pip install -q pystoi
    try:
        from pystoi import stoi as stoi_fn
        HAS_STOI = True
        print("STOI installed successfully")
    except:
        HAS_STOI = False
        print("STOI unavailable, skipping")

lm_out = os.path.join(LOCAL_OUTPUT, 'librimix')
results = []

before_files = sorted(os.listdir(os.path.join(lm_out, 'before')))
for fname in before_files:
    if not fname.endswith('.wav'):
        continue
    mix_id = fname.replace('.wav', '')

    before_wav, sr = torchaudio.load(os.path.join(lm_out, 'before', fname))
    after_wav, _ = torchaudio.load(os.path.join(lm_out, 'after_reuse', fname))
    gt_path = os.path.join(lm_out, 'clean_gt', fname)
    if not os.path.exists(gt_path):
        continue
    gt_wav, _ = torchaudio.load(gt_path)

    before_wav = before_wav.squeeze(0)
    after_wav = after_wav.squeeze(0)
    gt_wav = gt_wav.squeeze(0)

    min_len = min(before_wav.shape[-1], after_wav.shape[-1], gt_wav.shape[-1])
    before_t = before_wav[:min_len]
    after_t = after_wav[:min_len]
    gt_t = gt_wav[:min_len]

    row = {
        'mixture_id': mix_id,
        'sisdr_before': round(si_sdr(before_t, gt_t), 3),
        'sisdr_after_reuse': round(si_sdr(after_t, gt_t), 3),
    }

    if HAS_PESQ:
        try:
            row['pesq_before'] = round(pesq_fn(sr, gt_t.numpy(), before_t.numpy(), 'wb'), 3)
            row['pesq_after_reuse'] = round(pesq_fn(sr, gt_t.numpy(), after_t.numpy(), 'wb'), 3)
        except Exception as e:
            row['pesq_before'] = None
            row['pesq_after_reuse'] = None

    if HAS_STOI:
        try:
            row['stoi_before'] = round(stoi_fn(gt_t.numpy(), before_t.numpy(), sr, extended=False), 4)
            row['stoi_after_reuse'] = round(stoi_fn(gt_t.numpy(), after_t.numpy(), sr, extended=False), 4)
        except Exception as e:
            row['stoi_before'] = None
            row['stoi_after_reuse'] = None

    results.append(row)

csv_path = os.path.join(LOCAL_OUTPUT, 'librimix_reuse_metrics.csv')
if results:
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

print(f"\n{'='*60}")
print(f"  RE-USE Enhancement Results on LibriMix ({len(results)} samples)")
print(f"{'='*60}")

sisdrs_before = [r['sisdr_before'] for r in results]
sisdrs_after = [r['sisdr_after_reuse'] for r in results]
print(f"  SI-SDR Before (mixture):  mean={np.mean(sisdrs_before):.2f}  median={np.median(sisdrs_before):.2f}")
print(f"  SI-SDR After  (RE-USE):   mean={np.mean(sisdrs_after):.2f}  median={np.median(sisdrs_after):.2f}")
print(f"  SI-SDR Improvement:       {np.mean(sisdrs_after) - np.mean(sisdrs_before):+.2f} dB")

if HAS_PESQ:
    pesqs_b = [r['pesq_before'] for r in results if r.get('pesq_before') is not None]
    pesqs_a = [r['pesq_after_reuse'] for r in results if r.get('pesq_after_reuse') is not None]
    if pesqs_b:
        print(f"  PESQ Before:              mean={np.mean(pesqs_b):.3f}")
        print(f"  PESQ After:               mean={np.mean(pesqs_a):.3f}")
        print(f"  PESQ Improvement:         {np.mean(pesqs_a) - np.mean(pesqs_b):+.3f}")

if HAS_STOI:
    stois_b = [r['stoi_before'] for r in results if r.get('stoi_before') is not None]
    stois_a = [r['stoi_after_reuse'] for r in results if r.get('stoi_after_reuse') is not None]
    if stois_b:
        print(f"  STOI Before:              mean={np.mean(stois_b):.4f}")
        print(f"  STOI After:               mean={np.mean(stois_a):.4f}")
        print(f"  STOI Improvement:         {np.mean(stois_a) - np.mean(stois_b):+.4f}")

print(f"{'='*60}")
print(f"CSV saved: {csv_path}")


In [ ]:
import os, torch, torchaudio, numpy as np

dihard_out = os.path.join(LOCAL_OUTPUT, 'dihard')

print(f"{'='*60}")
print(f"  RE-USE Enhancement: DIHARD Recordings")
print(f"{'='*60}")

for rec_id in sorted(os.listdir(dihard_out)):
    rec_dir = os.path.join(dihard_out, rec_id)
    if not os.path.isdir(rec_dir):
        continue

    before, sr = torchaudio.load(os.path.join(rec_dir, 'before.wav'))
    after, _ = torchaudio.load(os.path.join(rec_dir, 'after_reuse.wav'))
    before = before.squeeze(0)
    after = after.squeeze(0)
    min_len = min(before.shape[-1], after.shape[-1])
    before = before[:min_len]
    after = after[:min_len]

    rms_before = torch.sqrt(torch.mean(before ** 2)).item()
    rms_after = torch.sqrt(torch.mean(after ** 2)).item()

    corr = torch.corrcoef(torch.stack([before, after]))[0, 1].item()

    dur = min_len / sr
    print(f"\n  {rec_id} ({dur:.0f}s):")
    print(f"    RMS before: {rms_before:.4f}")
    print(f"    RMS after:  {rms_after:.4f}")
    print(f"    Correlation (before vs after): {corr:.4f}")

print(f"\n{'='*60}")
print("No clean ground truth for DIHARD -> listen to before/after audio.")


In [ ]:
import os, torch, torchaudio

yt_out = os.path.join(LOCAL_OUTPUT, 'youtube')

before, sr = torchaudio.load(os.path.join(yt_out, 'before.wav'))
after, _ = torchaudio.load(os.path.join(yt_out, 'after_reuse.wav'))
before = before.squeeze(0)
after = after.squeeze(0)
min_len = min(before.shape[-1], after.shape[-1])
before = before[:min_len]
after = after[:min_len]

rms_before = torch.sqrt(torch.mean(before ** 2)).item()
rms_after = torch.sqrt(torch.mean(after ** 2)).item()
corr = torch.corrcoef(torch.stack([before, after]))[0, 1].item()

print(f"{'='*60}")
print(f"  RE-USE Enhancement: YouTube Documentary")
print(f"{'='*60}")
print(f"  Duration: {min_len/sr:.1f}s")
print(f"  RMS before: {rms_before:.4f}")
print(f"  RMS after:  {rms_after:.4f}")
print(f"  Correlation: {corr:.4f}")
print(f"{'='*60}")
print("Listen to before/after audio for qualitative comparison.")


In [ ]:
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/AD-FlowTSE_checkpoints/reuse_experiment_output'

if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)

shutil.copytree(LOCAL_OUTPUT, DRIVE_OUT)


total = 0
for root, dirs, files in os.walk(DRIVE_OUT):
    total += len(files)

print(f"Saved {total} files to {DRIVE_OUT}")
print(f"\nStructure:")
for root, dirs, files in os.walk(DRIVE_OUT):
    level = root.replace(DRIVE_OUT, '').count(os.sep)
    indent = '  ' * level
    n = len([f for f in files if f.endswith(('.wav', '.flac'))])
    n_csv = len([f for f in files if f.endswith('.csv')])
    parts = []
    if n > 0:
        parts.append(f"{n} audio")
    if n_csv > 0:
        parts.append(f"{n_csv} csv")
    suffix = f" ({', '.join(parts)})" if parts else ''
    print(f"{indent}{os.path.basename(root)}/{suffix}")

print(f"\nDownload this folder to your local machine.")
print(f"Place it at: AD-FlowTSE/reuse_experiment/output/")
